# Machine Learning Algorithms Laboratory - Experiment 2
## Email Spam/Ham Classification using Naïve Bayes and K-Nearest Neighbors (KNN)
**Student Name:** Rishi Rithesh  
**Register Number:** 3122247001049  
**Department:** M.Tech (Integrated) CSE - V Semester  
**Course Code:** ICS1512

### 1. Import Libraries and Load Dataset

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
)

plt.style.use('ggplot')
%matplotlib inline

# Load Dataset
df = pd.read_csv('spambase_csv.csv')
print('Raw Dataset Shape:', df.shape)
print('Target Distribution:\n', df['class'].value_counts())

### 2. Data Preprocessing & Cleaning

In [ ]:
# Missing value check
print('Missing Values:', df.isnull().sum().sum())

# Duplicate removal
num_duplicates = df.duplicated().sum()
print('Duplicate Rows:', num_duplicates)
df_clean = df.drop_duplicates().reset_index(drop=True)
print('Cleaned Shape:', df_clean.shape)

# Features and Target
X = df_clean.drop(columns=['class'])
y = df_clean['class']

### 3. Exploratory Data Analysis (Plots 1 - 4)

In [ ]:
# Plot 1: Class Distribution
class_counts = y.value_counts()
plt.figure(figsize=(6, 4))
bars = plt.bar(['Ham (0)', 'Spam (1)'], [class_counts[0], class_counts[1]], color=['#2b5c8f', '#d95f02'], width=0.5)
plt.title('Class Distribution: Spam (1) vs Ham (0)', fontweight='bold')
plt.ylabel('Number of Emails')
plt.show()

In [ ]:
# Plot 2: Correlation Heatmap
top_features = df_clean.corr()['class'].abs().sort_values(ascending=False).index[1:16]
top_corr = df_clean[list(top_features) + ['class']].corr()
plt.figure(figsize=(10, 8))
plt.matshow(top_corr, cmap='coolwarm', fignum=1)
plt.colorbar()
plt.xticks(range(len(top_corr.columns)), top_corr.columns, rotation=90, fontsize=8)
plt.yticks(range(len(top_corr.columns)), top_corr.columns, fontsize=8)
plt.title('Correlation Matrix of Top Features', pad=20, fontweight='bold')
plt.show()

### 4. Train-Test Split and Normalization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

scaler = MinMaxScaler()
X_train_minmax = scaler.fit_transform(X_train)
X_test_minmax = scaler.transform(X_test)

X_train_bin = (X_train_minmax > 0.05).astype(int)
X_test_bin = (X_test_minmax > 0.05).astype(int)
print(f'Train samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}')

### 5. Train & Evaluate Naïve Bayes Classifiers

In [ ]:
models_nb = {
    'Gaussian NB': (GaussianNB(), X_train_minmax, X_test_minmax),
    'Multinomial NB': (MultinomialNB(), X_train_minmax, X_test_minmax),
    'Bernoulli NB': (BernoulliNB(), X_train_bin, X_test_bin)
}

nb_results = []
for name, (model, X_tr, X_te) in models_nb.items():
    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    nb_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1-Score': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_prob)
    })

pd.DataFrame(nb_results)

### 6. KNN Classifier & Hyperparameter Tuning

In [ ]:
param_grid = {
    'n_neighbors': [1, 3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

grid = GridSearchCV(KNeighborsClassifier(algorithm='kd_tree'), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_minmax, y_train)
print('GridSearchCV Best Parameters:', grid.best_params_)
print('Best CV Accuracy:', grid.best_score_)
print('Test Accuracy:', grid.score(X_test_minmax, y_test))

### 7. 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
gnb_scores = cross_val_score(GaussianNB(), X_train_minmax, y_train, cv=cv)
mnb_scores = cross_val_score(MultinomialNB(), X_train_minmax, y_train, cv=cv)
bnb_scores = cross_val_score(BernoulliNB(), X_train_bin, y_train, cv=cv)
knn_scores = cross_val_score(grid.best_estimator_, X_train_minmax, y_train, cv=cv)

cv_df = pd.DataFrame({
    'Gaussian NB': gnb_scores,
    'Multinomial NB': mnb_scores,
    'Bernoulli NB': bnb_scores,
    'Best KNN': knn_scores
})
cv_df.loc['Average'] = cv_df.mean()
cv_df